# Lab 3D Gaussian Splatting · Colab T4

Chọn **Runtime → Change runtime type → T4 GPU**. Notebook dùng mã GraphDECO tại commit ghim và chạy từng mốc riêng. Đọc hướng dẫn trong `lab/README.md` trước khi chuẩn bị video và mask.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/3dgs_lab")
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/3dgs_lab")
WORK.mkdir(exist_ok=True)
VIDEO = DRIVE / "capture.mov"
DATA = WORK / "dataset"
UPSTREAM = WORK / "gaussian-splatting"
LAB = WORK / "lab"
BASELINE = DRIVE / "baseline"
OBJECT = DRIVE / "object"
ITERATIONS = 7000
RESOLUTION = 800
DENSIFY_UNTIL = 5000

def run(*args, **kwargs):
    subprocess.run([str(a) for a in args], check=True, **kwargs)

def unpack(archive, dest):
    dest.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archive), str(dest))

# Mặc định dùng ZIP chứa trực tiếp prepare.py, train_object.py và demo.py.
if not LAB.exists():
    unpack(DRIVE / "lab.zip", LAB)
assert (LAB / "prepare.py").is_file(), "lab.zip phải chứa trực tiếp các script lab."


## 1. Cài môi trường
Chạy lại sau khi đổi runtime. Nếu build lỗi, đối chiếu `torch.version.cuda` với phiên bản `nvcc` thực tế. Notebook dùng PyTorch của runtime và build các extension với toolkit của runtime.

In [ ]:
import torch
from torch.utils.cpp_extension import CUDA_HOME

run("nvidia-smi")
assert torch.cuda.is_available() and CUDA_HOME, "Cần GPU runtime có CUDA toolkit."
run(Path(CUDA_HOME) / "bin" / "nvcc", "--version")
print("PyTorch", torch.__version__, "CUDA", torch.version.cuda)
os.environ["CUDA_HOME"] = CUDA_HOME
os.environ["TORCH_CUDA_ARCH_LIST"] = ".".join(map(str, torch.cuda.get_device_capability()))
os.environ["MAX_JOBS"] = "2"
os.environ["QT_QPA_PLATFORM"] = "offscreen"

PIN = "54c035f7834b564019656c3e3fcc3646292f727d"
if not (UPSTREAM / ".git").exists():
    run("git", "clone", "--filter=blob:none", "https://github.com/graphdeco-inria/gaussian-splatting.git", UPSTREAM)
run("git", "checkout", "--detach", PIN, cwd=UPSTREAM)
run("git", "submodule", "update", "--init", "--recursive",
    "submodules/diff-gaussian-rasterization", "submodules/simple-knn", "submodules/fused-ssim", cwd=UPSTREAM)
run("apt-get", "-qq", "update")
run("apt-get", "-qq", "install", "-y", "colmap", "ffmpeg", "ninja-build")
run(sys.executable, "-m", "pip", "install", "plyfile", "ninja", "opencv-python-headless", "scipy")
run(sys.executable, "-m", "pip", "install", "--no-build-isolation",
    UPSTREAM / "submodules/diff-gaussian-rasterization",
    UPSTREAM / "submodules/simple-knn", UPSTREAM / "submodules/fused-ssim")
run(sys.executable, "-c", "import diff_gaussian_rasterization, simple_knn._C, fused_ssim")


## 2. COLMAP
Notebook lưu dataset để có thể tiếp tục ở phiên khác. Khi đổi video, dùng một thư mục Drive khác để tránh lấy lại dữ liệu cũ.

In [ ]:
dataset_archive = DRIVE / "dataset.zip"
if not (DATA / "sparse/0/images.bin").exists():
    if dataset_archive.exists():
        unpack(dataset_archive, DATA)
    else:
        run(sys.executable, LAB / "prepare.py", VIDEO, DATA, "--frames", 90)
        archive = Path(shutil.make_archive(str(WORK / "dataset"), "zip", DATA))
        shutil.copy2(archive, dataset_archive)

# Lưu đúng cách chia mặc định --eval của GraphDECO cho bộ dữ liệu này.
sys.path.insert(0, str(UPSTREAM))
from scene.colmap_loader import read_extrinsics_binary
names = sorted(camera.name for camera in read_extrinsics_binary(str(DATA / "sparse/0/images.bin")).values())
holdout = names[::8]
train = [name for name in names if name not in set(holdout)]
(DRIVE / "split.json").write_text(json.dumps({"train": train, "holdout": holdout}, indent=2))
print(len(train), "train,", len(holdout), "holdout")


## 3. Train và render baseline
Chạy thử 500 vòng trước. Các lượt train ghi mô hình và checkpoint trực tiếp vào Drive. Để tiếp tục sau khi runtime ngắt, điền đường dẫn tới checkpoint trong biến `RESUME` trước khi chạy lại cell tương ứng.

In [ ]:
def train_model(output, iterations, masked=False, resume=None):
    command = [sys.executable, UPSTREAM / "train.py"]
    if masked:
        command = [sys.executable, LAB / "train_object.py",
                   "--upstream", UPSTREAM, "--masks", WORK / "masks"]
    checkpoints = [i for i in (2000, 5000) if i < iterations] + [iterations]
    flags = ["-s", DATA, "-m", output, "--eval", "--disable_viewer", "--data_device", "cpu",
             "--resolution", RESOLUTION, "--iterations", iterations,
             "--densify_until_iter", min(DENSIFY_UNTIL, iterations),
             "--test_iterations", "-1", "--checkpoint_iterations", *checkpoints]
    if resume is not None:
        flags += ["--start_checkpoint", resume]
    run(*command, *flags, cwd=UPSTREAM)

def render_holdout(output):
    run(sys.executable, UPSTREAM / "render.py", "-s", DATA, "-m", output,
        "--iteration", ITERATIONS, "--skip_train", cwd=UPSTREAM)

train_model(WORK / "smoke", 500)


In [ ]:
RESUME = None  # Ví dụ BASELINE / "chkpnt5000.pth"
train_model(BASELINE, ITERATIONS, resume=RESUME)
render_holdout(BASELINE)


## 4. Train vật thể với mask
Tải `mask_inputs.zip` từ Drive, vẽ mask nhị phân trên ảnh đã khử méo rồi đưa `masks.zip` trở lại Drive. Mỗi mask PNG có tên cùng stem với ảnh và cùng kích thước. Chuẩn bị mask cho cả train và holdout để so sánh. Notebook không tự phân đoạn.

In [ ]:
archive = Path(shutil.make_archive(str(WORK / "mask_inputs"), "zip", DATA / "images"))
shutil.copy2(archive, DRIVE / "mask_inputs.zip")


In [ ]:
masks = WORK / "masks"
if not masks.exists():
    unpack(DRIVE / "masks.zip", masks)
missing = [name for name in names if not (masks / (Path(name).stem + ".png")).is_file()]
assert not missing, f"Thiếu mask (đặt PNG trực tiếp trong ZIP): {missing[:10]}"
RESUME = None  # Ví dụ OBJECT / "chkpnt5000.pth"
train_model(OBJECT, ITERATIONS, masked=True, resume=RESUME)
render_holdout(OBJECT)


## 5. Đánh giá và xuất demo
Ảnh so sánh gồm ảnh thật, baseline và mô hình tách vật thể. PSNR chỉ tính trên pixel thuộc mask. Hai video lần lượt kiểm tra góc camera mới và chuyển động của vật thể qua sau vật cản.

In [ ]:
import numpy as np
from PIL import Image

comparison = DRIVE / "comparison"
comparison.mkdir(exist_ok=True)
scores = []
for index, name in enumerate(holdout):
    image_name = f"{index:05d}.png"
    baseline = BASELINE / "test" / f"ours_{ITERATIONS}"
    object_run = OBJECT / "test" / f"ours_{ITERATIONS}"
    gt, base, obj = [Image.open(path).convert("RGB") for path in [
        baseline / "gt" / image_name, baseline / "renders" / image_name,
        object_run / "renders" / image_name]]
    with Image.open(masks / (Path(name).stem + ".png")) as label:
        mask = np.asarray(label.convert("L").resize(gt.size, Image.Resampling.NEAREST)) >= 128
    assert mask.any(), f"Mask rỗng sau resize: {name}"
    target = np.asarray(gt, dtype=float)[mask] / 255
    def psnr(image):
        mse = np.mean((np.asarray(image, dtype=float)[mask] / 255 - target) ** 2)
        return float(-10 * np.log10(max(mse, 1e-12)))
    scores.append({"frame": name, "baseline_psnr": psnr(base), "object_psnr": psnr(obj)})
    strip = Image.new("RGB", (3 * gt.width, gt.height))
    for column, image in enumerate([gt, base, obj]):
        strip.paste(image, (column * gt.width, 0))
    strip.save(comparison / image_name)
(comparison / "scores.json").write_text(json.dumps(scores, indent=2))


In [ ]:
run(sys.executable, LAB / "demo.py", "--upstream", UPSTREAM,
    "--model", OBJECT, "--iteration", ITERATIONS, "--output", DRIVE / "demo")
# Nếu vật thể bị nghiêng, chạy lại và thêm --up X Y Z trong hệ tọa độ COLMAP.


## 6. Viewer tương tác sau khi train

Có thể chạy cell này ngay sau khi có PLY baseline hoặc PLY vật thể. Nếu mở phiên Colab mới, chạy lại phần gắn Drive và cài môi trường trước. Chọn mô hình và vòng lặp, chỉnh camera rồi bấm **Render góc nhìn**. Viewer cần runtime GPU đang hoạt động.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from viewer import show_viewer

show_viewer(UPSTREAM, {"Baseline": BASELINE, "Vật thể": OBJECT})
